# Chapter 19 — Threshold models with feedback

Companion notebook to `docs/models/ch19_threshold_feedback.md` and the Streamlit page `app/pages/19_Threshold_Feedback.py`.

Same skeleton — *I watch what others are doing, then decide* — with two opposite dynamics depending on the sign of the feedback:
- **Granovetter (positive feedback)** — join if enough others act. Runs to extremes, outcomes are bimodal.
- **El Farol (negative feedback)** — act if few others do. Stabilises the average, randomises the individual case.

Plus **tipping points** — a small parameter shift flips the qualitative outcome. Signature: a spike in outcome entropy.

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

from modelthinker.dynamics.threshold_feedback import (
    cascade_variance_sweep,
    network_cascade,
    sample_uniform_thresholds,
    simulate_cascade,
    simulate_el_farol,
)

## 1. Granovetter's canonical pair

Two crowds with the same average threshold and completely different fates.

In [ ]:
# Crowd A: thresholds 0, 1, 2, ..., 99 (every rung of the ladder present)
crowd_a = sample_uniform_thresholds(n=100)
res_a   = simulate_cascade(crowd_a)

# Crowd B: change ONE threshold from 1 to 2 — average moves from 49.50 to 49.51
crowd_b = crowd_a.copy()
crowd_b[1] = 2.0
res_b   = simulate_cascade(crowd_b)

print(f'Crowd A · average threshold {crowd_a.mean():.2f}  ->  {res_a.final_size} people joined')
print(f'Crowd B · average threshold {crowd_b.mean():.2f}  ->  {res_b.final_size} people joined')
print(f'\nThe average moved by {crowd_b.mean() - crowd_a.mean():.3f}. The outcome changed by {res_a.final_size - res_b.final_size}.')

### The cumulative-threshold picture

Draw the cumulative threshold curve $F(x)$ = how many people have threshold $\le x$, and the 45° line. The cascade climbs the 45° line until $F(x)$ first drops below it — that's where it stops.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, thr, res, title in zip(
    axes, [crowd_a, crowd_b], [res_a, res_b], ['Crowd A · full riot', 'Crowd B · one loner'], strict=True,
):
    sorted_thr = np.sort(thr)
    xs = np.linspace(0, 100, 400)
    Fx = np.searchsorted(sorted_thr, xs, side='right')
    ax.plot(xs, Fx, color='#4C6EF5', lw=2.5, label='F(x)')
    ax.plot(xs, xs, color='#868E96', lw=1.2, ls=':', label='y = x')
    ax.plot(res.final_size, res.final_size, 'o', color='#F76707', ms=10, label=f'stops @ {res.final_size}')
    ax.set(title=title, xlabel='others acting', ylabel='people whose threshold is met', xlim=(0, 100), ylim=(0, 100))
    ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Variance matters more than the mean

Hold the mean threshold fixed at 25 and sweep the SD. Across the sweep, the *distribution of outcomes* is bimodal: mostly full riots or mostly fizzles, almost nothing in between. The average number of joiners is not a typical outcome.

In [ ]:
sds = np.array([2, 5, 10, 15, 20, 25, 40, 60, 100.0])
sweep = cascade_variance_sweep(mean_threshold=25.0, sds=sds, n_agents=100,
                                n_runs=200, instigators=1, seed=42)

print(f'{"SD":>5}  {"mean joiners":>13}  {"% full riot":>13}  {"% fizzle":>10}  {"entropy (bits)":>15}')
print('-' * 65)
for sd, m, ff, fz, h in zip(sweep.parameter_values, sweep.mean_outcome,
                              sweep.fraction_full, sweep.fraction_fizzle,
                              sweep.entropy_bits, strict=True):
    print(f'{sd:>5.1f}  {m:>13.1f}  {100*ff:>12.0f}%  {100*fz:>9.0f}%  {h:>15.3f}')

### The tipping-point signature

The **mean** cascade size rises smoothly with SD — hides the story. The **outcome entropy** spikes at the tip, where the same threshold distribution flips coin-fair between full cascade and fizzle.

In [ ]:
sds_fine = np.linspace(1, 60, 40)
sweep_fine = cascade_variance_sweep(mean_threshold=25.0, sds=sds_fine,
                                     n_agents=100, n_runs=200,
                                     instigators=1, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(sweep_fine.parameter_values, sweep_fine.mean_outcome,
             color='#4C6EF5', lw=2.5)
axes[0].set(xlabel='SD of threshold distribution',
            ylabel='mean cascade size', title='Smooth-looking curve')
axes[0].grid(alpha=0.3)

axes[1].plot(sweep_fine.parameter_values, sweep_fine.entropy_bits,
             color='#F76707', lw=2.5)
peak = np.argmax(sweep_fine.entropy_bits)
axes[1].axvline(sweep_fine.parameter_values[peak], color='#868E96', ls='--', lw=1)
axes[1].set(xlabel='SD of threshold distribution',
            ylabel='outcome entropy (bits)', title='The tipping-point signature')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Peak entropy at SD ≈ {sweep_fine.parameter_values[peak]:.1f}: '
      f'{sweep_fine.entropy_bits[peak]:.2f} bits '
      f'(max possible = {np.log2(10):.2f} bits with 10 bins)')

## 3. Network cascades (Watts 2002)

Real thresholds are usually fractions of *your neighbours*, not counts of the whole population. On a network you get Duncan Watts' famous **cascade window** — global cascades exist only in a middle band of network density.

In [ ]:
n = 500
mean_degrees = [1, 2, 3, 4, 6, 8, 12, 20]
phi = 0.18   # each node acts if 18% of neighbours have
runs = 50
print(f'{"<k>":>4}  {"mean cascade":>14}  {"% global (>50%)":>18}')
print('-' * 42)
for k in mean_degrees:
    sizes = []
    for r in range(runs):
        G = nx.erdos_renyi_graph(n, k / (n - 1), seed=r)
        res = network_cascade(G, threshold_frac=phi, seed=r)
        sizes.append(res.final_size / n)
    sizes = np.array(sizes)
    print(f'{k:>4}  {100*sizes.mean():>13.1f}%  {100*(sizes > 0.5).mean():>17.0f}%')

## 4. El Farol — negative feedback

Same rule shape, opposite sign. The catastrophic failure mode is *homogeneity of belief*, not homogeneity of preference.

In [ ]:
r_hom = simulate_el_farol(n_agents=100, capacity=60, n_weeks=100,
                            homogeneous=True, seed=7)
r_div = simulate_el_farol(n_agents=100, capacity=60, n_weeks=100,
                            homogeneous=False, predictors_per_agent=3, seed=7)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ax, r, name, color in zip(
    axes, [r_hom, r_div],
    ['Everyone uses the SAME predictor', 'Diverse predictors'],
    ['#E03131', '#4C6EF5'], strict=True,
):
    ax.plot(np.arange(1, 101), r.attendance, color=color, lw=1.8)
    ax.axhline(60, color='#868E96', ls='--', lw=1)
    ax.set(ylabel='attendance', title=f'{name}  ·  mean {r.mean:.1f}, SD {r.std:.1f}',
           ylim=(0, 100))
    ax.grid(alpha=0.3)
axes[1].set_xlabel('week')
plt.tight_layout(); plt.show()

## 5. What-ifs to try

1. **Sweep instigators.** Run `cascade_variance_sweep` with `instigators = 0, 1, 2, 3, 5`. Where does the entropy peak move? How many instigators does it take to turn a fizzle regime into a cascade regime?
2. **Reproduce Watts' cascade window.** Sweep mean degree from 1 to 20 (via ER edge probability) at fixed $\phi = 0.18$. Plot the global-cascade rate. You should see a bell-shaped curve peaking around ⟨k⟩ ≈ 3.
3. **El Farol with a *tiny* predictor pool.** Restrict the predictor pool to just two rules (say `_pred_last_week` and `_pred_mirror`). Does diversity still stabilise attendance?
4. **Delayed negative feedback.** Modify the El Farol logic so agents react to attendance from *two* weeks ago instead of last week. Watch stabilisation flip into oscillation. That's Ch 18's systems-dynamics point arriving in threshold clothing.
5. **Bank runs as a threshold model.** Interpret the cascade as a bank run: threshold = 'how many others withdrawing I'd need to see before I withdraw'. What levers move the outcome? (Answer: change payoffs so the threshold is never reached — deposit insurance.)